In [1]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import argparse
import numpy as np
import random
import scipy
import scipy.sparse as sp
import networkx as nx
import matplotlib.pyplot as plt
import random
from scipy.stats import kendalltau
import pandas as pd

In [2]:
def MI(res):
    a=pd.DataFrame(res.detach().numpy())
    x=len(a)
    a.rank(axis=0,method='min',numeric_only=None,
    na_option='keep',ascending=True,pct=False)
    b=a.iloc[:,0].value_counts()
    y=0
    for i in range(len(b)):
        y=y+b.iloc[i]*(b.iloc[i]-1)
    ans=(1-y/(x*(x-1)))*(1-y/(x*(x-1)))
    return ans

In [3]:
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
def ERM(G):
    max_number = 0
    for node in G.nodes():   #计算的最大dk2
        neighbors = list(G.neighbors(node))     #节点i的一阶邻居
        di2_max=0
        for node1 in neighbors:
            neighbors1 = list(G.neighbors(node1))     #节点j的一阶邻居
            temp2 = sum(G.degree(neigh) for neigh in neighbors1)
            di2_max = di2_max + temp2
        if di2_max>max_number:
            max_number=di2_max
    # print(max_number)
    length=len(G.nodes())
    my_list_EC=[]
    my_list_SI=[]
    my_list_ERM = []
    for _ in range(length):
        my_list_EC.append(None)
        my_list_SI.append(None)
        my_list_ERM.append(None)
    for node in G.nodes():
        neighbors = list(G.neighbors(node))          #节点i的一阶邻居
        di1 = sum(G.degree(neigh) for neigh in neighbors)
        E1=0
        di2=0
        for node1 in neighbors:
            dj=G.degree(node1)
            temp1=-(dj/di1)*np.log(dj/di1)
            E1=E1+temp1
            neighbors1=list(G.neighbors(node1))
            temp2=sum(G.degree(neigh) for neigh in neighbors1)
            di2=di2+temp2
        # print(di1,di2)
        #print(node)
        # print(E1)
        E2=0
        for node2 in neighbors:
            neighbors2 = list(G.neighbors(node2))  #节点j的一阶邻居
            dj1 = sum(G.degree(neigh) for neigh in neighbors2)
            temp3=-(dj1/di2)*np.log(dj1/di2)
            E2=E2+temp3
        # print(E2)
        lamuda_node=di2/max_number
        # print(lamuda_node)
        EC_node=E1+lamuda_node*E2
        my_list_EC[node]=EC_node
    # print(my_list_EC)
    for node in G.nodes():
        neighbors3=list(G.neighbors(node))
        SI_node=0
        for t in neighbors3:
            SI_node=SI_node+my_list_EC[t]
        my_list_SI[node]=SI_node
    for node in G.nodes():
        neighbors4=list(G.neighbors(node))
        ERM_node=0
        for t in neighbors4:
            ERM_node=ERM_node+my_list_SI[t]
        my_list_ERM[node]=ERM_node
    return  my_list_ERM

In [4]:
def calculate_top_k_jaccard(list1, list2,length):
    """
    计算两个列表按影响力排序后不同Top-K集合的Jaccard相似度
    
    参数:
    list1, list2: 节点影响力列表，下标为节点ID，值为影响力
    k_list: 需要计算的K值列表，默认为[100, 200, 300]
    
    返回:
    字典，键为K，值为对应Jaccard相似度
    """
    # 将列表转换为NumPy数组
    k_list = [int(length*0.05),int(length*0.06),int(length*0.07),int(length*0.08),int(length*0.09),int(length*0.10),int(length*0.11),int(length*0.12),int(length*0.13),int(length*0.14),int(length*0.15),int(length*0.16),int(length*0.17),int(length*0.18),int(length*0.19),int(length*0.20),int(length*0.21),int(length*0.22),int(length*0.23),int(length*0.24),int(length*0.25),int(length*0.26),int(length*0.27),int(length*0.28),int(length*0.29),int(length*0.30),int(length*0.31),int(length*0.32),int(length*0.33),int(length*0.34),int(length*0.35),int(length*0.36),int(length*0.37),int(length*0.38),int(length*0.39),int(length*0.40),int(length*0.41),int(length*0.42),int(length*0.43),int(length*0.44),int(length*0.45),int(length*0.46),int(length*0.47),int(length*0.48),int(length*0.49),int(length*0.50)]
    array1 = np.array(list1)
    array2 = np.array(list2)
    
    # 对数组排序并获取节点ID（argsort默认升序，加负号转为降序）
    sorted_indices1 = np.argsort(-array1)  # array1中影响力从大到小的节点ID
    # print(sorted_indices1)
    sorted_indices2 = np.argsort(-array2)  # array2中影响力从大到小的节点ID
    # print(sorted_indices2)
    # 计算各Top-K的Jaccard相似度
    results = {}
    for k in k_list:
        # 取前k个节点ID形成集合
        top_k_set1 = set(sorted_indices1[:k])
        top_k_set2 = set(sorted_indices2[:k])
        
        # 计算交集和并集大小
        intersection = len(top_k_set1 & top_k_set2)
        union = len(top_k_set1 | top_k_set2)
        
        # 计算Jaccard相似度
        jaccard = intersection / union if union > 0 else 0
        results[k] = jaccard
    
    return results

In [5]:
def kshell(G):
    """
    kshell(G)计算k-shell值
    """
    graph = G.copy()
    importance_dict = {}
    ks = 1
    while graph.nodes():
        temp = []
        node_degrees_dict = gDegree(graph)
        #print(node_degrees_dict)
        kks = min(node_degrees_dict.values())
        if(kks<=ks):
            while True:
                for k, v in node_degrees_dict.items():
                    if v <= ks:
                        temp.append(k)
                        graph.remove_node(k)
                        node_degrees_dict = gDegree(graph)
                if len(graph.nodes()) == 0:
                    break
                else:
                    t=min(node_degrees_dict.values())
                if t>ks :
                    break
                #if kks not in node_degrees_dict.values():
                #   break
            importance_dict[ks] = temp
        ks += 1
    return importance_dict

def gDegree(G):
    """
    将G.degree()的返回值变为字典
    """
    node_degrees_dict = {}
    for i in G.degree():
        node_degrees_dict[i[0]] = i[1]
    return node_degrees_dict.copy()

def LSS(G):
    temp_G=G
    ks=kshell(temp_G)
    num_nodes=len(temp_G.nodes())
    shell_numbers = np.zeros(num_nodes, dtype=float)

    for ks1, nodes in ks.items():
        for node in nodes:
            shell_numbers[node] = ks1
    # print(shell_numbers)
    weight=[]
    for i in range(num_nodes):
        node_degree=temp_G.degree(i)
        weight.append(node_degree*shell_numbers[i])

    # print(weight)


    edg = G.edges()
    adj_matrix = [[0 for _ in range(num_nodes)] for _ in range(num_nodes)]
    adj_matrix = np.array(adj_matrix)
    # print(edg)
    for edg1 in edg:
        adj_matrix[edg1[0]][edg1[1]] = 1
        adj_matrix[edg1[1]][edg1[0]] = 1


    cf_matrix = [[0 for _ in range(num_nodes)] for _ in range(num_nodes)]
    tw_matrix = [[0 for _ in range(num_nodes)] for _ in range(num_nodes)]
    for i in range(num_nodes):
        for j in range(num_nodes):
            if adj_matrix[i][j]==1:
                # 获得相连的两个节点的共同邻居，因为不相连的两个节点不会构成三角形
                neighbors1 = set(G.neighbors(i))
                neighbors2 = set(G.neighbors(j))
                # 找出共同邻居节点
                common_neighbors = neighbors1.intersection(neighbors2)
                #构成的三角形的个数等于共同邻居的节点个数。
                a=len(common_neighbors)
                cf_matrix[i][j]=(shell_numbers[i]-a)*(shell_numbers[j]-a)
                tw_matrix[i][j]=cf_matrix[i][j]*weight[i]*weight[j]/(a+1)
            # else:
            #     cf_matrix[i][j]=(shell_numbers[i]-0)*(shell_numbers[j]-0)
            #     tw_matrix=cf_matrix[i][j]*weight[i]*weight[j]
    # print(cf_matrix[10][1])
    # print(tw_matrix[10][1])
    LSS_matrix=[]
    for i in range(num_nodes):
        sum=0
        nei = G.neighbors(i)
        for j in nei:
            sum = sum + tw_matrix[i][j]
        LSS_matrix.append(shell_numbers[i] * sum / num_nodes)

    return LSS_matrix

In [265]:
def h_index_centrality(graph):
    """
    计算H指数中心性
    :param graph: 无向图，使用字典表示，键为节点，值为与该节点相连的节点列表
    :return: H指数中心性字典，键为节点，值为H指数中心性值
    """
    nodes = []
    for node in graph.nodes():
        nodes.append(node)
    # print(nodes)

    n = len(nodes)
    h_index = {node: 0 for node in nodes}
    for i, node in enumerate(nodes):
        degrees = [len(graph[n]) for n in graph[node]]
        degrees.sort(reverse=True)
        for j, degree in enumerate(degrees):
            if degree >= j + 1:
                h_index[node] = j + 1
            else:
                break
    # h_centrality = {node: h_index[node] / n for node in nodes} 归一化处理
    h_centrality = {node: h_index[node] for node in nodes}

    return h_centrality

In [23]:
edges=np.genfromtxt("facebook_combined.txt",dtype=int)
edges=edges
print(edges.shape)
adj = sp.coo_matrix((np.ones(edges.shape[0]), (edges[:, 0], edges[:, 1])),
                        shape=(4039, 4039))
adj = adj + adj.T.multiply(adj.T > adj) - adj.multiply(adj.T > adj)
adj_facebook = torch.FloatTensor(np.array(adj.todense()))
adj=np.array(adj_facebook)
G_facebook = nx.Graph()
G_facebook = nx.from_numpy_array(adj) 
list_facebook=list(range(4039))

(88234, 2)


In [269]:
lable_facebook=np.load("lable/lable_facebook_1.0.npy")
lable_facebook_t=torch.tensor(lable_facebook).float()
node_feature_facebook=torch.load("EC做损失的特征向量/node_feature_facebook_LSTM_EC_0.005_500.pt")
node_feature_facebook = min_max_normalization(node_feature_facebook)

In [271]:
start=time.time()
facebook_ERM = ERM(G_facebook)
end=time.time()
print(end-start)
print(MI(torch.tensor(facebook_ERM)))
print("----------------")
for i in range(10, 21):
    t=i/10
    if t!=1.5:
        lable_facebook=np.load("lable/lable_facebook_"+str(t)+".npy")
    else:
        lable_facebook=np.load("lable/lable_facebook.npy")
    lable_facebook_t=torch.tensor(lable_facebook).float()
    tau,_=kendalltau(facebook_ERM,lable_facebook_t)
    print(tau)
print("------------------")
lable_facebook=np.load("lable/lable_facebook.npy")
lable_facebook_t=torch.tensor(lable_facebook).float()
length=len(facebook_ERM)
k_list = [int(length*0.05),int(length*0.1),int(length*0.15),int(length*0.2),int(length*0.25),int(length*0.3),int(length*0.35),int(length*0.4),int(length*0.45),int(length*0.5)]
similarities = calculate_top_k_jaccard(facebook_ERM, lable_facebook_t,length)
for k, sim in similarities.items():
    print(sim)

21.54754400253296
0.9998803186027777
----------------
0.7540547421180964
0.7519849326008965
0.7578953109273064
0.7541268926489904
0.7623486048316126
0.7658034655030043
0.7770184808368139
0.7814415591696676
0.7899962494672771
0.8006936354081516
0.805670643136221
------------------
0.8440366972477065
0.7794117647058824
0.7195121951219512
0.7226666666666667
0.7203791469194313
0.7675438596491229
0.8159509202453987
0.8579654510556622
0.8918918918918919
0.9120135363790186
0.936
0.9457831325301205
0.9351198871650211
0.8712998712998713
0.8240190249702735
0.7834254143646409
0.7502579979360166
0.7192642787996127
0.7011915673693859
0.6735751295336787
0.6622734761120264
0.6496465043205027
0.6403310759969902
0.6424418604651163
0.6423562412342216
0.6498637602179836
0.6582781456953642
0.6681730148482892
0.6860759493670886
0.6929716399506781
0.6942446043165468
0.7095825984714874
0.7172413793103448
0.7206954570947841
0.7241379310344828
0.726349545697488
0.7275574112734864
0.7323799795709908
0.736
0.747

In [273]:
start=time.time()
facebook_lss = LSS(G_facebook)
end=time.time()
print(end-start)
print(MI(torch.tensor(facebook_lss)))
print("----------------")
for i in range(10, 21):
    t=i/10
    if t!=1.5:
        lable_facebook=np.load("lable/lable_facebook_"+str(t)+".npy")
    else:
        lable_facebook=np.load("lable/lable_facebook.npy")
    lable_facebook_t=torch.tensor(lable_facebook).float()
    tau,_=kendalltau(facebook_lss,lable_facebook_t)
    print(tau)
print("-----------")
lable_facebook=np.load("lable/lable_facebook.npy")
lable_facebook_t=torch.tensor(lable_facebook).float()
length=len(facebook_lss)
k_list = [int(length*0.05),int(length*0.1),int(length*0.15),int(length*0.2),int(length*0.25),int(length*0.3),int(length*0.35),int(length*0.4),int(length*0.45),int(length*0.5)]
similarities = calculate_top_k_jaccard(facebook_lss, lable_facebook_t,length)
for k, sim in similarities.items():
    # print(f"Top-{k} Jaccard Similarity: {sim:.4f}")
    print(sim)

8.111369848251343
0.9998729613735187
----------------
0.5541167014623871
0.5466703164393591
0.5424826558777021
0.5412526815331082
0.5357370127259291
0.5358975485669782
0.5354257472525991
0.5344339725684554
0.5419973728785638
0.5457785946498814
0.5522199683715718
-----------
0.6144578313253012
0.6187290969899666
0.6686390532544378
0.7180851063829787
0.7621359223300971
0.7675438596491229
0.7831325301204819
0.8026070763500931
0.8103448275862069
0.8284789644012945
0.8558282208588958
0.8643578643578643
0.8565629228687416
0.8289308176100629
0.798358733880422
0.7775330396475771
0.7430626927029804
0.7109826589595376
0.696526508226691
0.6721311475409836
0.6677685950413224
0.64576802507837
0.6256524981357197
0.6142857142857143
0.6118375774260152
0.6135909393737509
0.6123631680618158
0.6160100062539087
0.6174863387978142
0.6210153482880756
0.6232050545663412
0.6245810055865921
0.6230309614340033
0.6189973614775726
0.6228748068006182
0.6166166166166166
0.6130604288499025
0.6098718557190318
0.61563

In [275]:
start=time.time()
eigenvector_centrality = nx.eigenvector_centrality(G_facebook,max_iter=1000,tol=3e-10)
# eigenvector_centrality = nx.eigenvector_centrality_numpy(G_facebook)
# print(centrality)
facebook_ec=[0 for _ in range(nums_nodes)]
print(nums_nodes) 
for i,j in eigenvector_centrality.items():
    facebook_ec[i]=j
end=time.time()
print(end-start)
print(MI(torch.tensor(facebook_ec)))
print("--------------------")
for i in range(10, 21):
    t=i/10
    if t!=1.5:
        lable_facebook=np.load("lable/lable_facebook_"+str(t)+".npy")
    else:
        lable_facebook=np.load("lable/lable_facebook.npy")
    lable_facebook_t=torch.tensor(lable_facebook).float()
    tau,_=kendalltau(facebook_ec,lable_facebook_t)
    print(tau)
print("------------------")
lable_facebook=np.load("lable/lable_facebook.npy")
lable_facebook_t=torch.tensor(lable_facebook).float()
similarities = calculate_top_k_jaccard(facebook_ec, lable_facebook_t,length)
for k, sim in similarities.items():
    print(sim)

4039
1.422020673751831
0.999877375707826
--------------------
0.5378759253742434
0.5523543384132668
0.5771662776192865
0.5892394482648116
0.6049839577459954
0.6112183641341555
0.620882573907556
0.6220391397519329
0.617406037160026
0.6172694449040195
0.6158460049770642
------------------
0.8525345622119815
0.7472924187725631
0.6987951807228916
0.6692506459948321
0.6537585421412301
0.6055776892430279
0.5578947368421052
0.5220125786163522
0.4914772727272727
0.4637305699481865
0.43705463182897863
0.42290748898678415
0.42176165803108806
0.4240940254652302
0.43498596819457436
0.4672727272727273
0.4722222222222222
0.5050847457627119
0.5364238410596026
0.5679611650485437
0.5902285263987391
0.6042780748663101
0.6172106824925816
0.631768953068592
0.6343335659455688
0.628782784129119
0.6228127025275437
0.6180338134001252
0.6343558282208589
0.6403823178016727
0.6565064478311841
0.669345579793341
0.6758272574312956
0.6783369803063457
0.6981132075471698
0.7053854276663146
0.7141377524598653
0.720953

In [279]:
start=time.time()
degrees = G_facebook.degree()
facebook_degree=[0 for _ in range(nums_nodes)]
for node,degree in degrees:
    facebook_degree[node]=degree
end=time.time()
print(end-start)
print(MI(torch.tensor(facebook_degree)))
print("-----------------------")
for i in range(10, 21):
    t=i/10
    if t!=1.5:
        lable_facebook=np.load("lable/lable_facebook_"+str(t)+".npy")
    else:
        lable_facebook=np.load("lable/lable_facebook.npy")
    lable_facebook_t=torch.tensor(lable_facebook).float()
    tau,_=kendalltau(facebook_degree,lable_facebook_t)
    print(tau)
print("------------------")
lable_facebook=np.load("lable/lable_facebook.npy")
lable_facebook_t=torch.tensor(lable_facebook).float()
similarities = calculate_top_k_jaccard(facebook_degree, lable_facebook_t,length)
for k, sim in similarities.items():
    print(sim)

0.002000093460083008
0.9739038649913806
-----------------------
0.6795650215937316
0.6600894556026226
0.6397115307931146
0.631452444848319
0.6209336628163531
0.6192093321208092
0.6163633021409843
0.6172068990239743
0.6251461818861077
0.6271221652073736
0.6304167549416604
------------------
0.5343511450381679
0.5868852459016394
0.6588235294117647
0.6735751295336787
0.704225352112676
0.7259100642398287
0.7584158415841584
0.7664233576642335
0.761744966442953
0.751937984496124
0.7163120567375887
0.6801040312093628
0.6550060313630881
0.6227678571428571
0.5979166666666667
0.5870206489675516
0.574744661095636
0.5675198587819947
0.5649241146711635
0.5454545454545454
0.5487336914811972
0.5463917525773195
0.5439093484419264
0.5395095367847411
0.5418038183015141
0.5555555555555556
0.5581829495955196
0.5670103092783505
0.5726092089728453
0.5763490241102182
0.5823068309070548
0.592552026286966
0.5961538461538461
0.597084851639771
0.6071428571428571
0.6077650572424091
0.613846903949293
0.61986628462

In [281]:
start=time.time()
h_index=h_index_centrality(G_facebook)
facebook_hindex=[0 for _ in range(nums_nodes)]
for node,hindex in h_index.items():
    facebook_hindex[node]=hindex
end=time.time()
print(end-start)
print(MI(torch.tensor(facebook_hindex)))
print("---------------")
for i in range(10, 21):
    t=i/10
    if t!=1.5:
        lable_facebook=np.load("lable/lable_facebook_"+str(t)+".npy")
    else:
        lable_facebook=np.load("lable/lable_facebook.npy")
    lable_facebook_t=torch.tensor(lable_facebook).float()
    tau,_=kendalltau(facebook_hindex,lable_facebook_t)
    print(tau)
print("------------------")
lable_facebook=np.load("lable/lable_facebook.npy")
lable_facebook_t=torch.tensor(lable_facebook).float()
similarities = calculate_top_k_jaccard(facebook_hindex, lable_facebook_t,length)
for k, sim in similarities.items():
    print(sim)

0.1535043716430664
0.966470931900529
---------------
0.7053656927815629
0.6866176105192552
0.6677417482248033
0.6602110475280419
0.650461453744788
0.6495889772843136
0.647789398194549
0.6487165581352258
0.6563987855018116
0.6592813515010877
0.663895778191293
------------------
0.7946428571428571
0.7163120567375887
0.6835820895522388
0.7226666666666667
0.7578692493946732
0.7911111111111111
0.8538622129436325
0.8796116504854369
0.9090909090909091
0.9152542372881356
0.8701700154559505
0.806993006993007
0.7657657657657657
0.7166469893742621
0.6710239651416122
0.6486210418794689
0.6292026897214217
0.6086956521739131
0.5917667238421955
0.573051948051948
0.5790297339593115
0.5849056603773585
0.5751445086705202
0.5771109560362875
0.5867208672086721
0.5986798679867987
0.6030729833546735
0.6139912554653342
0.6174863387978142
0.6181496758986447
0.619484240687679
0.6236739251814629
0.6318951392681594
0.6301806588735388
0.631279129984464
0.6288451840645487
0.6297390448055146
0.639439342677622
0.646

In [11]:
def min_max_normalization(tensor):
    # 找出数组的最小值和最大值
    min_val = torch.min(tensor)
    max_val = torch.max(tensor)

    # 进行最小 - 最大标准化
    normalized_tensor = (tensor - min_val) / (max_val - min_val)
    return normalized_tensor
# node_feature_BA = min_max_normalization(node_feature_BA)
# node_feature_BA1 = min_max_normalization(node_feature_BA1)

In [13]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# 假设你的特征矩阵叫 node_feature_vidal
X = node_feature_facebook.detach().numpy()

# 1. 肘部法则（Elbow Method）
sse = []
k_list = range(2, 15)  # 尝试K从2到14
for k in k_list:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X)
    sse.append(kmeans.inertia_)  # inertia_就是SSE

plt.figure()
plt.plot(k_list, sse, marker='o')
plt.xlabel('Number of clusters K')
plt.ylabel('SSE (Inertia)')
plt.title('Elbow Method for Choosing K')
plt.show()

NameError: name 'node_feature_facebook' is not defined

In [ ]:
import numpy as np
from sklearn.cluster import KMeans
# 假设 node_feature_jazz 是你的节点特征矩阵，形状是 (num_nodes, feature_dim)
random.seed(17)
np.random.seed(17)
torch.manual_seed(17)
    
# 设置聚类数量
k = 5
X = node_feature_facebook.detach().numpy()

# 执行 KMeans 聚类
kmeans = KMeans(n_clusters=k, random_state=42)
labels = kmeans.fit_predict(X)

# labels 是每个节点对应的聚类类别，比如 labels[i] 是第i个节点所属的簇编号

# 将节点按簇分类整理
cluster_nodes = {i: [] for i in range(k)}
for idx, label in enumerate(labels):
    cluster_nodes[label].append(idx)
#选择训练集的节点
n_nodes = node_feature_facebook.shape[0]

# 1. 随机采样 5% 的节点索引
num_train = int(0.1 * n_nodes)+1
split_idx=int(num_train * 0.5)
sum_idx = []
while len(sum_idx) < num_train:
    for i in range(k):
        if cluster_nodes[i]:  # 确保当前簇中还有节点可取
            node = random.choice(cluster_nodes[i])  # 随机选一个
            sum_idx.append(node)
            cluster_nodes[i].remove(node)
            if len(sum_idx) >= num_train:
                break
print(sum_idx)
train_idx=sum_idx[:split_idx]
pinggu_idx=sum_idx[split_idx:]
print(train_idx)
print(pinggu_idx)
# 2. 构造训练集（只选取部分节点特征）
train_facebook = node_feature_facebook[train_idx]
train_facebook_lable = lable_facebook_t[train_idx]
# 3. 测试集就是整个 node_feature_BA
pinggu_facebook = node_feature_facebook
pinggu_facebook_lable = lable_facebook_t[pinggu_idx]
sum_facebook_lable=lable_facebook_t[sum_idx]

In [15]:
class GNN(torch.nn.Module):
    def __init__(self,input_feature,output_feature):
        super(GNN,self).__init__()
        self.w = nn.Parameter(torch.empty(size=(input_feature,output_feature)))
        self.a= nn.Parameter(torch.empty(size=(1,output_feature)))
        self.sigmod=torch.nn.Sigmoid()
        self.reset_parameters()
    def reset_parameters(self):
        #nn.init.xavier_uniform_(self.w.data,gain=1.414)
        #nn.init.xavier_uniform_(self.a.data,gain=1.414)
        for param in self.parameters():
             nn.init.xavier_uniform_(param)
        
    def forward(self,x,adj):
        adj=torch.Tensor(adj.numpy()+np.identity(adj.shape[0]))
        # adj=torch.FloatTensor(normalize_adj(sp.csr_matrix(adj)+ sp.eye(adj.shape[0])).todense())
        x=torch.mm(adj,x)
        x=torch.mm(x,self.w)
        x=x.add(self.a)
        x=torch.relu(x)
        return x


class CGNN(torch.nn.Module):
    def __init__(self):
        super(CGNN,self).__init__()
        self.layer2=GNN(48,6)
        self.layer3=GNN(6,12)
        # self.fc1=torch.nn.Linear(6,12)
        self.fc1=torch.nn.Linear(12,1)
        # self.fc2=torch.nn.Linear(24,1)
        
        
       
     
    
    
    def forward(self,x,adj,target_nodes):
        #x=self.layer1(x)
        # x=torch.cat([x1,x],dim=-1)
        # x=0.5*x+0.5*x1
        x=self.layer2(x,adj)
        x=self.layer3(x,adj)
       # x=self.layer4(x,adj)
        
        x=x[target_nodes]
        
        x=self.fc1(x.view(x.size(0),-1))
        x=torch.relu(x)
        # x=self.fc2(x)
        # x=torch.relu(x)
        x=x.flatten()
        
        return x

In [141]:
import optuna
import time
import numpy as np
import torch
import torch.optim as optim
from scipy.stats import kendalltau

# 假设你的模型和数据已定义
# from your_model import CGNN  # 导入你的CGNN模型
# 假设以下变量已提前定义：
# node_feature_facebook, adj_facebook, train_idx, list_facebook, list_pinggu
# train_facebook_lable, pinggu_true  # 训练标签和验证集真实值

def objective(trial):
    # 1. 定义超参数搜索空间
    learning_rate = trial.suggest_float(
        "learning_rate", 
        low=0.001, 
        high=0.01, 
        step=0.001  # 学习率：0.001-0.01，步长0.001
    )
    epochs = trial.suggest_int(
        "epochs", 
        low=100, 
        high=1000, # 训练次数：100-1000之间的任意整数
        step=100
    )

    epoch_1=trial.suggest_int(
        "epoch_1", 
        low=100, 
        high=500, # 训练次数：100-1000之间的任意整数
        step=100
    )
    learning_rate1 = trial.suggest_categorical(
        "learning_rate1", 
        [0.005,0.001]  # 仅允许这三个候选值
    )
    # 2. 初始化模型和优化器

    lable_facebook=np.load("lable/lable_facebook.npy")
    lable_facebook_t=torch.tensor(lable_facebook).float()
    node_feature_facebook=torch.load("EC做损失的特征向量/node_feature_facebook_LSTM_EC_"+str(learning_rate1)+"_"+str(epoch_1)+".pt")
    node_feature_facebook = min_max_normalization(node_feature_facebook)
    
    random.seed(17)
    np.random.seed(17)
    torch.manual_seed(17)
    
# 设置聚类数量
    k = 6
    X = node_feature_facebook.detach().numpy()

# 执行 KMeans 聚类
    kmeans = KMeans(n_clusters=k, random_state=42)
    labels = kmeans.fit_predict(X)

# labels 是每个节点对应的聚类类别，比如 labels[i] 是第i个节点所属的簇编号

# 将节点按簇分类整理
    cluster_nodes = {i: [] for i in range(k)}
    for idx, label in enumerate(labels):
        cluster_nodes[label].append(idx)
#选择训练集的节点
    n_nodes = node_feature_facebook.shape[0]

# 1. 随机采样 5% 的节点索引
    # 1. 随机采样 5% 的节点索引
    num_train = int(0.05 * n_nodes)+1

    train_idx=[]
    while len(train_idx) < num_train:
        for i in range(k):
            if cluster_nodes[i]:  # 确保当前簇中还有节点可取
                node = random.choice(cluster_nodes[i])  # 随机选一个
                train_idx.append(node)
                cluster_nodes[i].remove(node)
                if len(train_idx) >= num_train:
                    break

    pinggu_idx=[]
    while len(pinggu_idx) < num_train:
        for i in range(k):
            if cluster_nodes[i]:  # 确保当前簇中还有节点可取
                node = random.choice(cluster_nodes[i])  # 随机选一个
                pinggu_idx.append(node)
                cluster_nodes[i].remove(node)
                if len(pinggu_idx) >= num_train:
                    break

# 2. 构造训练集（只选取部分节点特征）
    train_facebook = node_feature_facebook[train_idx]
    train_facebook_lable = lable_facebook_t[train_idx]
# 3. 测试集就是整个 node_feature_BA
    
    pinggu_facebook = node_feature_facebook[pinggu_idx]
    pinggu_facebook_lable = lable_facebook_t[pinggu_idx]


    
    # 2. 初始化模型和优化器
    random.seed(17)
    np.random.seed(17)
    torch.manual_seed(17)
    model = CGNN()  # 实例化你的模型
    optimizer = optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=5e-4  # 保持权重衰减不变
    )
    
    # 3. 训练模型
    for epoch in range(epochs):
        # 训练过程（复用你的train函数逻辑）
        model.train()
        output = model(node_feature_facebook.data, adj_facebook, train_idx)
        loss_train = torch.nn.functional.mse_loss(output, train_facebook_lable)
        optimizer.zero_grad()
        loss_train.backward()
        optimizer.step()
        
        # 每100轮打印一次中间结果（可选）
        if (epoch + 1) % 100 == 0:
            print(f"当前参数组合: 学习率={learning_rate}, 训练次数={epochs}, 轮次={epoch+1}, 训练损失={loss_train.item():.4f}")
    
    # 4. 在验证集上评估RMSE
    model.eval()
    with torch.no_grad():  # 关闭梯度计算，节省内存
        pinggu_pre = model(node_feature_facebook.data, adj_facebook, pinggu_idx)
        # 转换为numpy数组计算RMSE
        pinggu_pre_np = pinggu_pre.detach().numpy()
        pinggu_true=pinggu_facebook_lable.detach().numpy()
        # print(kendalltau(pinggu_pre_np,pinggu_pre_np))
        rmse=kendalltau(pinggu_pre_np,pinggu_true).statistic
        # rmse = np.sqrt(np.mean((pinggu_true - pinggu_pre_np) **2))
    
    # 5. 返回需要最小化的RMSE（Optuna会寻找最小RMSE对应的参数）
    return rmse

if __name__ == '__main__':
    # 创建优化实例，方向为"最小化"RMSE
    study = optuna.create_study(direction="maximize")
    # 尝试20组参数组合
    study.optimize(objective, n_trials=50)
    
    # 输出最优结果
    print("\n优化完成！")
    print(f"最佳参数组合: {study.best_params}")
    print(f"最小RMSE: {study.best_value:.4f}")
    print(f"最佳试验编号: {study.best_trial.number}")

[I 2025-09-18 16:46:35,585] A new study created in memory with name: no-name-4a77c85c-7402-495f-8e5a-8cf1f68aa7bc
C:\Users\Administrator\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=7.
  warnings.warn(


当前参数组合: 学习率=0.008, 训练次数=1000, 轮次=100, 训练损失=30.5046
当前参数组合: 学习率=0.008, 训练次数=1000, 轮次=200, 训练损失=29.0393
当前参数组合: 学习率=0.008, 训练次数=1000, 轮次=300, 训练损失=26.8330
当前参数组合: 学习率=0.008, 训练次数=1000, 轮次=400, 训练损失=25.2689
当前参数组合: 学习率=0.008, 训练次数=1000, 轮次=500, 训练损失=23.6899
当前参数组合: 学习率=0.008, 训练次数=1000, 轮次=600, 训练损失=22.8087
当前参数组合: 学习率=0.008, 训练次数=1000, 轮次=700, 训练损失=22.0599
当前参数组合: 学习率=0.008, 训练次数=1000, 轮次=800, 训练损失=21.0056
当前参数组合: 学习率=0.008, 训练次数=1000, 轮次=900, 训练损失=20.2979


[I 2025-09-18 16:47:29,834] Trial 0 finished with value: 0.7989782886334611 and parameters: {'learning_rate': 0.008, 'epochs': 1000, 'epoch_1': 200, 'learning_rate1': 0.005}. Best is trial 0 with value: 0.7989782886334611.


当前参数组合: 学习率=0.008, 训练次数=1000, 轮次=1000, 训练损失=18.6636


C:\Users\Administrator\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=7.
  warnings.warn(


...

当前参数组合: 学习率=0.006, 训练次数=700, 轮次=700, 训练损失=16.9794


C:\Users\Administrator\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=7.
  warnings.warn(


当前参数组合: 学习率=0.003, 训练次数=300, 轮次=100, 训练损失=74.4378
当前参数组合: 学习率=0.003, 训练次数=300, 轮次=200, 训练损失=31.9112


[I 2025-09-18 17:13:55,443] Trial 49 finished with value: 0.8306513680934214 and parameters: {'learning_rate': 0.003, 'epochs': 300, 'epoch_1': 500, 'learning_rate1': 0.001}. Best is trial 7 with value: 0.8401687087058478.


当前参数组合: 学习率=0.003, 训练次数=300, 轮次=300, 训练损失=29.5274

优化完成！
最佳参数组合: {'learning_rate': 0.003, 'epochs': 1000, 'epoch_1': 300, 'learning_rate1': 0.001}
最小RMSE: 0.8402
最佳试验编号: 7


In [17]:
node_feature_facebook=torch.load("EC做损失的特征向量/node_feature_facebook_LSTM_EC_0.005_100.pt")
node_feature_facebook = min_max_normalization(node_feature_facebook)
lable_facebook=np.load("lable/lable_facebook.npy")
lable_facebook_t=torch.tensor(lable_facebook).float()

In [19]:
import numpy as np
from sklearn.cluster import KMeans
# 假设 node_feature_facebook 是你的节点特征矩阵，形状是 (num_nodes, feature_dim)
random.seed(17)
np.random.seed(17)
torch.manual_seed(17)
    
# 设置聚类数量
k = 5
X = node_feature_facebook.detach().numpy()

# 执行 KMeans 聚类
kmeans = KMeans(n_clusters=k, random_state=42)
labels = kmeans.fit_predict(X)

# labels 是每个节点对应的聚类类别，比如 labels[i] 是第i个节点所属的簇编号

# 将节点按簇分类整理
cluster_nodes = {i: [] for i in range(k)}
for idx, label in enumerate(labels):
    cluster_nodes[label].append(idx)
#选择训练集的节点
n_nodes = node_feature_facebook.shape[0]

# 1. 随机采样 5% 的节点索引
num_train = int(0.05 * n_nodes)+1

train_idx=[]
while len(train_idx) < num_train:
    for i in range(k):
        if cluster_nodes[i]:  # 确保当前簇中还有节点可取
            node = random.choice(cluster_nodes[i])  # 随机选一个
            train_idx.append(node)
            cluster_nodes[i].remove(node)
            if len(train_idx) >= num_train:
                break

pinggu_idx=[]
while len(pinggu_idx) < num_train:
    for i in range(k):
        if cluster_nodes[i]:  # 确保当前簇中还有节点可取
            node = random.choice(cluster_nodes[i])  # 随机选一个
            pinggu_idx.append(node)
            cluster_nodes[i].remove(node)
            if len(pinggu_idx) >= num_train:
                break

# 2. 构造训练集（只选取部分节点特征）
train_facebook = node_feature_facebook[train_idx]
train_facebook_lable = lable_facebook_t[train_idx]
# 3. 测试集就是整个 node_feature_BA
    
pinggu_facebook = node_feature_facebook[pinggu_idx]
pinggu_facebook_lable = lable_facebook_t[pinggu_idx]

In [25]:
import time
import numpy as np
import torch
import torch.optim as optim
from scipy.stats import kendalltau

# 确保随机种子完全一致
random.seed(17)
np.random.seed(17)
torch.manual_seed(17)
# 对于CUDA环境，还需固定cuda随机种子
if torch.cuda.is_available():
    torch.cuda.manual_seed(17)
    torch.cuda.manual_seed_all(17)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def train(epoch, model, optimizer, node_feature, adj, train_idx, train_label):
    model.train()
    output = model(node_feature.data, adj, train_idx)
    loss = torch.nn.functional.mse_loss(output, train_label)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.item()

if __name__ == '__main__':
    # 确保数据与Optuna调优时完全一致
    # （需与objective函数中使用的node_feature_facebook、adj_facebook等保持相同）
    # 假设以下变量已正确定义：
    # node_feature_facebook, adj_facebook, train_idx, train_facebook_lable, sum_idx, sum_facebook_lable
    
    # 初始化模型和优化器（与Optuna中完全一致）
    model = CGNN()  # 重新初始化，确保权重初始状态一致
    optimizer = optim.Adam(
        model.parameters(),
        lr=0.001,  # Optuna找到的最佳学习率
        weight_decay=5e-4
    )
    
    # 记录总训练时间
    t_total = time.time()
    loss_values = []
    
    # 训练600轮（与Optuna最佳参数一致）
    for epoch in range(400):
        loss = train(epoch, model, optimizer, node_feature_facebook, adj_facebook, train_idx, train_facebook_lable)
        loss_values.append(loss)
        
        # 每100轮打印训练损失（与Optuna中保持一致的监控频率）
        if (epoch + 1) % 100 == 0:
            print(f"Epoch {epoch+1}, 训练损失: {loss:.4f}")
    
    # 关键：添加与Optuna中完全一致的评估步骤
    model.eval()
    with torch.no_grad():
        # 使用相同的评估索引和标签
        pinggu_pre = model(node_feature_facebook.data, adj_facebook, pinggu_idx)
        pinggu_pre_np = pinggu_pre.detach().numpy()
        pinggu_true = pinggu_facebook_lable.detach().numpy()
        # 计算Kendall系数（与Optuna中评估指标一致）
        kendall = kendalltau(pinggu_pre_np, pinggu_true).statistic
        print(f"最终Kendall系数: {kendall:.6f}")
    
    print("Optimization Finished!")
    print(f"Total time elapsed: {time.time() - t_total:.2f}s")

Epoch 100, 训练损失: 1087.6500
Epoch 200, 训练损失: 1010.6700
Epoch 300, 训练损失: 921.7166
Epoch 400, 训练损失: 825.6727
最终Kendall系数: 0.861682
Optimization Finished!
Total time elapsed: 115.75s


In [33]:
lable_facebook=np.load("lable/lable_facebook.npy")
lable_facebook_t=torch.tensor(lable_facebook).float()

In [39]:
model_cgnn=model
model_cgnn.eval()
start=time.time()
output = model_cgnn(node_feature_facebook.data,adj_facebook,list_facebook)

for i in train_idx:
    output[i]=lable_facebook_t[i]
for i in pinggu_idx:
    output[i]=lable_facebook_t[i]
end=time.time()
print(end-start)
output_rank=output.detach().numpy().argsort()

# print(output_rank)
# print(label_rank)
loss_train = torch.nn.functional.mse_loss(output,lable_facebook_t)
# print(loss_train)
print(kendalltau(output.detach().numpy(),lable_facebook_t))
print(MI(output))

0.2504861354827881
SignificanceResult(statistic=0.793602802154962, pvalue=0.0)
0.9999602689059347


In [28]:
length=len(output)
k_list = [int(length*0.05),int(length*0.1),int(length*0.15),int(length*0.2),int(length*0.25),int(length*0.3),int(length*0.35),int(length*0.4),int(length*0.45),int(length*0.5)]
similarities = calculate_top_k_jaccard(output.detach().numpy(), lable_facebook_t,length)
for k, sim in similarities.items():
    print(sim)

0.8190045248868778
0.7728937728937729
0.7407407407407407
0.7412398921832885
0.7536231884057971
0.7831858407079646
0.8309278350515464
0.8687258687258688
0.8987341772151899
0.9217687074829932
0.9267515923566879
0.9546142208774584
0.9628040057224606
0.9490616621983914
0.9151061173533084
0.8811188811188811
0.8495092693565977
0.8122448979591836
0.7932367149758454
0.7602179836512262
0.7517361111111112
0.729818780889621
0.7165354330708661
0.7018072289156626
0.6995645863570392
0.7032348804500703
0.7138945927446955
0.7215189873417721
0.7309941520467836
0.7412809131261889
0.7422934648581998
0.7486470234515935
0.7555816686251469
0.76220562894888
0.761744966442953
0.7650273224043715
0.7662753468516542
0.7759162303664922
0.7823408624229979
0.7868275515334339
0.7839960726558665
0.7855769230769231
0.79649787032655
0.7927844588344126
0.8007279344858963
0.8115746971736204


In [ ]:
# 0.005/100  0.001/400  5聚簇   0.7908926345134847